# PCS221 Cloud Computing – Lab Assignment 6
## MapReduce (Without EMR)
### Each question solved using:
- **(A) Manual Python** (Mapper → Shuffle → Reducer)
- **(B) mrjob Framework**

In [ ]:
# Install mrjob if not available
!pip install mrjob -q

---
## Q1. Word Count
**Task:** Count the frequency of each word  
**Input:** `hadoop is fast` / `hadoop is scalable`

In [ ]:
# ── Q1 (A): Manual MapReduce ──
from collections import defaultdict

q1_input = ["hadoop is fast", "hadoop is scalable"]

# Mapper
def q1_mapper(lines):
    pairs = []
    for line in lines:
        for word in line.strip().split():
            pairs.append((word.lower(), 1))
    return pairs

# Shuffle
def shuffle(pairs):
    grouped = defaultdict(list)
    for key, value in pairs:
        grouped[key].append(value)
    return dict(grouped)

# Reducer
def sum_reducer(grouped):
    return {k: sum(v) for k, v in grouped.items()}

mapped   = q1_mapper(q1_input)
shuffled = shuffle(mapped)
result   = sum_reducer(shuffled)

print("Q1 (A) – Word Count:")
for word, count in sorted(result.items()):
    print(f"  {word}: {count}")

In [ ]:
%%writefile q1_wordcount.py
from mrjob.job import MRJob

class MRWordCount(MRJob):
    def mapper(self, _, line):
        for word in line.strip().split():
            yield word.lower(), 1

    def reducer(self, word, counts):
        yield word, sum(counts)

if __name__ == '__main__':
    MRWordCount.run()

In [ ]:
# ── Q1 (B): mrjob ──
q1_text = "hadoop is fast\nhadoop is scalable"
with open('q1_input.txt', 'w') as f:
    f.write(q1_text)

!python q1_wordcount.py q1_input.txt

---
## Q2. Character Count
**Task:** Count the frequency of each character (ignore spaces)  
**Input:** `big data`

In [ ]:
# ── Q2 (A): Manual MapReduce ──
q2_input = ["big data"]

def q2_mapper(lines):
    pairs = []
    for line in lines:
        for ch in line:
            if ch != ' ':
                pairs.append((ch, 1))
    return pairs

mapped   = q2_mapper(q2_input)
shuffled = shuffle(mapped)
result   = sum_reducer(shuffled)

print("Q2 (A) – Character Count:")
for ch, count in sorted(result.items()):
    print(f"  '{ch}': {count}")

In [ ]:
%%writefile q2_charcount.py
from mrjob.job import MRJob

class MRCharCount(MRJob):
    def mapper(self, _, line):
        for ch in line:
            if ch != ' ':
                yield ch, 1

    def reducer(self, ch, counts):
        yield ch, sum(counts)

if __name__ == '__main__':
    MRCharCount.run()

In [ ]:
# ── Q2 (B): mrjob ──
with open('q2_input.txt', 'w') as f:
    f.write('big data')

!python q2_charcount.py q2_input.txt

---
## Q3. Average Word Length (Per Word)
**Task:** Compute the average length of each unique word  
**Input:** `data science data big data`

In [ ]:
# ── Q3 (A): Manual MapReduce ──
q3_input = ["data science data big data"]

def q3_mapper(lines):
    pairs = []
    for line in lines:
        for word in line.strip().split():
            pairs.append((word.lower(), len(word)))
    return pairs

def avg_reducer(grouped):
    return {k: sum(v) / len(v) for k, v in grouped.items()}

mapped   = q3_mapper(q3_input)
shuffled = shuffle(mapped)
result   = avg_reducer(shuffled)

print("Q3 (A) – Average Word Length (Per Word):")
for word, avg in sorted(result.items()):
    print(f"  {word}: {avg:.2f}")

In [ ]:
%%writefile q3_avgwordlen.py
from mrjob.job import MRJob

class MRAvgWordLen(MRJob):
    def mapper(self, _, line):
        for word in line.strip().split():
            yield word.lower(), len(word)

    def reducer(self, word, lengths):
        lengths = list(lengths)
        yield word, sum(lengths) / len(lengths)

if __name__ == '__main__':
    MRAvgWordLen.run()

In [ ]:
# ── Q3 (B): mrjob ──
with open('q3_input.txt', 'w') as f:
    f.write('data science data big data')

!python q3_avgwordlen.py q3_input.txt

---
## Q4. Global Average Word Length
**Task:** Compute the average length of ALL words across the entire input  
**Input:** `hadoop mapreduce spark`

In [ ]:
# ── Q4 (A): Manual MapReduce ──
q4_input = ["hadoop mapreduce spark"]

def q4_mapper(lines):
    pairs = []
    for line in lines:
        for word in line.strip().split():
            pairs.append(("global", len(word)))
    return pairs

mapped   = q4_mapper(q4_input)
shuffled = shuffle(mapped)
result   = avg_reducer(shuffled)

print("Q4 (A) – Global Average Word Length:")
for key, avg in result.items():
    print(f"  {avg:.2f}")

In [ ]:
%%writefile q4_globalavg.py
from mrjob.job import MRJob

class MRGlobalAvgWordLen(MRJob):
    def mapper(self, _, line):
        for word in line.strip().split():
            yield 'global', (len(word), 1)

    def reducer(self, key, values):
        total_len, total_count = 0, 0
        for length, count in values:
            total_len   += length
            total_count += count
        yield key, total_len / total_count

if __name__ == '__main__':
    MRGlobalAvgWordLen.run()

In [ ]:
# ── Q4 (B): mrjob ──
with open('q4_input.txt', 'w') as f:
    f.write('hadoop mapreduce spark')

!python q4_globalavg.py q4_input.txt

---
## Q5. Q1–Q4 on External File + Top 5 Most Frequent Words
**Note:** Download the file from Google Drive and upload it as `q5_input.txt` in your working directory.

The cell below uses `gdown` to auto-download it.

In [ ]:
!pip install gdown -q
!gdown --id 16TIgKhcc2JH8jyJXfwouOV3y7ACX_aas -O q5_input.txt
print("File downloaded. First 300 chars:")
with open('q5_input.txt') as f:
    print(f.read(300))

In [ ]:
# ── Q5 (A): Manual MapReduce on external file ──
with open('q5_input.txt') as f:
    q5_lines = f.readlines()

# Word Count
wc_mapped   = q1_mapper(q5_lines)
wc_shuffled = shuffle(wc_mapped)
wc_result   = sum_reducer(wc_shuffled)
print(f"Total unique words: {len(wc_result)}")

# Top 5 most frequent words
top5 = sorted(wc_result.items(), key=lambda x: x[1], reverse=True)[:5]
print("\nTop 5 Most Frequent Words:")
for word, count in top5:
    print(f"  {word}: {count}")

# Character Count
cc_mapped   = q2_mapper(q5_lines)
cc_shuffled = shuffle(cc_mapped)
cc_result   = sum_reducer(cc_shuffled)
print(f"\nUnique chars: {len(cc_result)}")

# Avg Word Length per Word
aw_mapped   = q3_mapper(q5_lines)
aw_shuffled = shuffle(aw_mapped)
aw_result   = avg_reducer(aw_shuffled)
print(f"Avg word length computed for {len(aw_result)} unique words.")

# Global Avg Word Length
ga_mapped   = q4_mapper(q5_lines)
ga_shuffled = shuffle(ga_mapped)
ga_result   = avg_reducer(ga_shuffled)
print(f"\nGlobal Average Word Length: {ga_result['global']:.4f}")

In [ ]:
%%writefile q5_top5.py
from mrjob.job import MRJob
from mrjob.step import MRStep

class MRTop5Words(MRJob):
    def steps(self):
        return [
            MRStep(mapper=self.mapper_count, reducer=self.reducer_count),
            MRStep(mapper=self.mapper_swap,  reducer=self.reducer_top5)
        ]

    def mapper_count(self, _, line):
        for word in line.strip().split():
            yield word.lower(), 1

    def reducer_count(self, word, counts):
        yield word, sum(counts)

    def mapper_swap(self, word, count):
        yield None, (count, word)

    def reducer_top5(self, _, pairs):
        top = sorted(pairs, key=lambda x: x[0], reverse=True)[:5]
        for count, word in top:
            yield word, count

if __name__ == '__main__':
    MRTop5Words.run()

In [ ]:
# ── Q5 (B): mrjob – Top 5 words ──
!python q5_top5.py q5_input.txt

---
## Q6. Average Marks Per Student
**Input:**
```
A 80
B 70
A 90
B 60
A 100
```

In [ ]:
# ── Q6 (A): Manual MapReduce ──
q6_input = ["A 80", "B 70", "A 90", "B 60", "A 100"]

def q6_mapper(lines):
    pairs = []
    for line in lines:
        parts = line.strip().split()
        student, marks = parts[0], int(parts[1])
        pairs.append((student, marks))
    return pairs

mapped   = q6_mapper(q6_input)
shuffled = shuffle(mapped)
result   = avg_reducer(shuffled)

print("Q6 (A) – Average Marks Per Student:")
for student, avg in sorted(result.items()):
    print(f"  Student {student}: {avg:.2f}")

In [ ]:
%%writefile q6_avgmarks.py
from mrjob.job import MRJob

class MRAvgMarks(MRJob):
    def mapper(self, _, line):
        parts = line.strip().split()
        if len(parts) == 2:
            student, marks = parts[0], int(parts[1])
            yield student, marks

    def reducer(self, student, marks):
        marks = list(marks)
        yield student, sum(marks) / len(marks)

if __name__ == '__main__':
    MRAvgMarks.run()

In [ ]:
# ── Q6 (B): mrjob ──
with open('q6_input.txt', 'w') as f:
    f.write("A 80\nB 70\nA 90\nB 60\nA 100")

!python q6_avgmarks.py q6_input.txt

---
## Q7. Average Salary Per Department + Highest Paid Department
**Input:**
```
HR 50000
IT 70000
HR 60000
IT 80000
```

In [ ]:
# ── Q7 (A): Manual MapReduce ──
q7_input = ["HR 50000", "IT 70000", "HR 60000", "IT 80000"]

def q7_mapper(lines):
    pairs = []
    for line in lines:
        parts = line.strip().split()
        dept, salary = parts[0], int(parts[1])
        pairs.append((dept, salary))
    return pairs

mapped   = q7_mapper(q7_input)
shuffled = shuffle(mapped)
result   = avg_reducer(shuffled)

print("Q7 (A) – Average Salary Per Department:")
for dept, avg in sorted(result.items()):
    print(f"  {dept}: {avg:.2f}")

highest = max(result, key=result.get)
print(f"\nHighest Paid Department: {highest} (avg: {result[highest]:.2f})")

In [ ]:
%%writefile q7_avgsalary.py
from mrjob.job import MRJob
from mrjob.step import MRStep

class MRAvgSalary(MRJob):
    def steps(self):
        return [
            MRStep(mapper=self.mapper,          reducer=self.reducer_avg),
            MRStep(mapper=self.mapper_find_max, reducer=self.reducer_max)
        ]

    def mapper(self, _, line):
        parts = line.strip().split()
        if len(parts) == 2:
            yield parts[0], int(parts[1])

    def reducer_avg(self, dept, salaries):
        salaries = list(salaries)
        avg = sum(salaries) / len(salaries)
        yield dept, avg

    def mapper_find_max(self, dept, avg):
        yield None, (avg, dept)

    def reducer_max(self, _, pairs):
        best = max(pairs, key=lambda x: x[0])
        yield 'Highest Paid Department', best[1]

if __name__ == '__main__':
    MRAvgSalary.run()

In [ ]:
# ── Q7 (B): mrjob ──
with open('q7_input.txt', 'w') as f:
    f.write("HR 50000\nIT 70000\nHR 60000\nIT 80000")

!python q7_avgsalary.py q7_input.txt

---
## Q8. Average Temperature Per Country
**Input:**
```
New York,38
London,29
Tokyo,35
New York,32
Delhi,45
Ambala,35
```

In [ ]:
# ── Q8 (A): Manual MapReduce ──
q8_input = [
    "New York,38",
    "London,29",
    "Tokyo,35",
    "New York,32",
    "Delhi,45",
    "Ambala,35"
]

def q8_mapper(lines):
    pairs = []
    for line in lines:
        parts = line.strip().rsplit(',', 1)
        if len(parts) == 2:
            city, temp = parts[0].strip(), float(parts[1].strip())
            pairs.append((city, temp))
    return pairs

mapped   = q8_mapper(q8_input)
shuffled = shuffle(mapped)
result   = avg_reducer(shuffled)

print("Q8 (A) – Average Temperature Per City:")
for city, avg in sorted(result.items()):
    print(f"  {city}: {avg:.2f}")

In [ ]:
%%writefile q8_avgtemp.py
from mrjob.job import MRJob

class MRAvgTemp(MRJob):
    def mapper(self, _, line):
        parts = line.strip().rsplit(',', 1)
        if len(parts) == 2:
            try:
                city = parts[0].strip()
                temp = float(parts[1].strip())
                yield city, temp
            except ValueError:
                pass

    def reducer(self, city, temps):
        temps = list(temps)
        yield city, sum(temps) / len(temps)

if __name__ == '__main__':
    MRAvgTemp.run()

In [ ]:
# ── Q8 (B): mrjob ──
q8_text = """New York,38
London,29
Tokyo,35
New York,32
Delhi,45
Ambala,35"""
with open('q8_input.txt', 'w') as f:
    f.write(q8_text)

!python q8_avgtemp.py q8_input.txt

---
## Q9. Average Temperature Per Country – Kaggle Dataset
**Dataset:** [Global Land Temperatures – Kaggle](https://www.kaggle.com/datasets/heemalichaudhari/global-land-temperatures)

Download `GlobalLandTemperaturesByCountry.csv` from Kaggle and place it in the working directory.

In [ ]:
# ── Q9: Check dataset ──
import os
import pandas as pd

dataset_path = 'GlobalLandTemperaturesByCountry.csv'

if not os.path.exists(dataset_path):
    print("Dataset not found. Please download from Kaggle and place in the current directory.")
    print("Expected file: GlobalLandTemperaturesByCountry.csv")
else:
    df = pd.read_csv(dataset_path)
    print(df.head())
    print(df.shape)

In [ ]:
# ── Q9 (A): Manual MapReduce on Kaggle dataset ──
if os.path.exists(dataset_path):
    df = pd.read_csv(dataset_path)
    df = df.dropna(subset=['AverageTemperature', 'Country'])

    # Prepare lines: Country,Temperature
    q9_lines = [f"{row['Country']},{row['AverageTemperature']}" for _, row in df.iterrows()]

    mapped   = q8_mapper(q9_lines)
    shuffled = shuffle(mapped)
    result   = avg_reducer(shuffled)

    print("Q9 (A) – Average Temperature Per Country (Top 10):")
    for country, avg in sorted(result.items(), key=lambda x: x[1], reverse=True)[:10]:
        print(f"  {country}: {avg:.2f}")
else:
    print("Skipping Q9 (A) — dataset not available.")

In [ ]:
%%writefile q9_prepare_input.py
import pandas as pd

df = pd.read_csv('GlobalLandTemperaturesByCountry.csv')
df = df.dropna(subset=['AverageTemperature', 'Country'])

with open('q9_input.txt', 'w') as f:
    for _, row in df.iterrows():
        f.write(f"{row['Country']},{row['AverageTemperature']}\n")

print("q9_input.txt written with", len(df), "rows.")

In [ ]:
# ── Q9 (B): mrjob on Kaggle dataset ──
if os.path.exists(dataset_path):
    !python q9_prepare_input.py
    !python q8_avgtemp.py q9_input.txt
else:
    print("Skipping Q9 (B) — dataset not available.")